In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/dias-rapids-25.02/lib/python3.10/site-packages/IPython/core/extensions.py:145: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/annotated/checkpoints/post_cell_9.pickle

trying: ['pd']
me:  0
trying: ['map_payment_type']
me:  17
trying: ['time']
me:  0
trying: ['trip_data']


me:  17
trying: ['file_loc']
me:  1
trying: ['orig_output']
me:  20
trying: ['sns']
me:  0
trying: ['plt']
me:  0


Declaring variable pd
Declaring variable time
Declaring variable sns
Declaring variable plt
Declaring variable file_loc
Declaring variable map_payment_type
Declaring variable trip_data
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 10 ###

# Sum the tax columns in a single GPU kernel
cols = ['extra','mta_tax','improvement_surcharge']
trip_data['total_taxes'] = trip_data[cols].sum(axis=1)
# Drop the now-redundant columns
t_trip_data = trip_data.drop(columns=cols, inplace=True)
# Display the result
trip_data.head()

CPU times: user 131 ms, sys: 23.3 ms, total: 154 ms
Wall time: 157 ms


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,duration,trip_pickup_hour,trip_dropoff_hour,trip_day,total_taxes
0,2018-01-01 00:21:05,2018-01-01 00:24:23,1,0.5,41,24,Cash,4.5,0.00,0.0,5.80,3.300000,0,0,Monday,1.3
1,2018-01-01 00:44:55,2018-01-01 01:03:05,1,2.7,239,140,Cash,14.0,0.00,0.0,15.30,18.166667,0,1,Monday,1.3
2,2018-01-01 00:08:26,2018-01-01 00:14:21,2,0.8,262,141,Credit_card,6.0,1.00,0.0,8.30,5.916667,0,0,Monday,1.3
3,2018-01-01 00:20:22,2018-01-01 00:52:51,1,10.2,140,257,Cash,33.5,0.00,0.0,34.80,32.483333,0,0,Monday,1.3
4,2018-01-01 00:09:18,2018-01-01 00:27:06,2,2.5,246,239,Credit_card,12.5,2.75,0.0,16.55,17.800000,0,0,Monday,1.3


In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/rewritten/o4_mini_high/checkpoints/post_cell_10_try_1.pickle

migration speed (bps): 793727331.751765
---------------------------
variables to migrate:
t_trip_data 16
file_loc 113
time 72
trip_data 48
plt 72
pd 72
cols 88
orig_output 16
map_payment_type 144
sns 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/nyc-taxi/rewritten/o4_mini_high/checkpoints/post_cell_10_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['trip_data']
Intermediate variables ['file_loc']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
  trip_data
    Input columns: set()
    Changed columns: set()
    Created columns: set()
    Deleted columns: {'RatecodeID', 'VendorID', 'store_and_fwd_flag'}
======= Cell 4 =======
Input variables []
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
  trip_data
    Input columns: set()
    Changed columns: {'tpep_pickup_datetime', 'tpep_dropoff_datetime'}
    Created columns: set()

In [7]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-taxi/opt_cell_exec_info_10_try_1.pkl", "wb"
) as f:
    pickle.dump(opt_cell_exec_info[10], f)

In [ ]:
opt_output = Out.get(4)

In [ ]:
trip_data_opt = trip_data
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/annotated/checkpoints/post_cell_10.pickle
assert compare_df(trip_data_opt, trip_data)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output